In [ ]:
import pandas as pd
from google.colab import userdata

data_path = userdata.get('DATA_PATH')
df = pd.read_csv(data_path)

df['week'] = pd.to_datetime(df['week'])

In [ ]:
WARMUP_WEEKS = 12

df2 = df[df['history_weeks'] >= WARMUP_WEEKS].copy()

In [ ]:
FEATURES = ['zero_ratio_12', 'demand_rate', 'q75_nonzero_demand', 'mean_nonzero_demand',
            'std_nonzero_demand', 'median_nonzero_demand', 'croston_alpha_demand',
            'croston_alpha_interval', 'croston_forecast', 'mean_demand_interval',
            'rolling_mean_12', 'rolling_std_12', 'ema_slow', 'ema_fast', 'macd',
            'sales_ratio', 'rolling_mean_4', 'rolling_std_4', 'recent_trend_4',
            'sbc_category', 'sin_1', 'sin_2', 'cos_1', 'adi', 'cv2']

FINAL_DECISION = {
    'Erratic':'model',
    'Lumpy': 'model',
    'LowData': 'baseline',
    'Intermittent': 'baseline',
    'Smooth': 'model'
}

In [ ]:
TEST_START = pd.Timestamp("2025-09-01")
TEST_END   = pd.Timestamp("2025-12-15")

INTERNAL_VAL_WEEKS = 8
INTERNAL_VAL_START = TEST_START - pd.Timedelta(weeks=INTERNAL_VAL_WEEKS)

In [ ]:
final_train = df2[df2['week'] < INTERNAL_VAL_START].copy()
internal_val = df2[(df2['week'] >= INTERNAL_VAL_START) & (df2['week'] < TEST_START)].copy()
test_df = df2[(df2['week'] >= TEST_START) & (df2['week'] <= TEST_END)].copy()

In [ ]:
final_train['sbc_category'] = final_train['sbc_category'].astype('category')
cat_categories = final_train['sbc_category'].cat.categories
internal_val['sbc_category'] = pd.Categorical(internal_val['sbc_category'], categories=cat_categories)

X_ftrain, y_ftrain = final_train[FEATURES], final_train['quantity']
X_ival, y_ival     = internal_val[FEATURES], internal_val['quantity']

In [ ]:
import lightgbm as lgb

probe_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.5,
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
probe_model.fit(
    X_ftrain, y_ftrain,
    eval_metric='tweedie',
    eval_set=[(X_ival, y_ival)],
    categorical_feature=['sbc_category'],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
)

BEST_ITERATION = probe_model.best_iteration_
print(f"Bulunan optimal ağaç sayısı: {BEST_ITERATION}")

#Başlangıçta en iyi n_estimators değerini bulmamız için veri setini zaman eksenin de train, val ve test olarak ayırdık.
#92 değerinin anlamı model artık 92. ağaçtan sonra hataları minimize etmiyor tam tersi ezberlemeye yöneliyor.

Bulunan optimal ağaç sayısı: 92


In [ ]:
full_train = df2[df2['week'] < TEST_START].copy()
full_train['sbc_category'] = full_train['sbc_category'].astype('category')
cat_categories_final = full_train['sbc_category'].cat.categories

X_full, y_full = full_train[FEATURES], full_train['quantity']

final_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.5,
    n_estimators=92,   # <-- sabit, CV'den bulunan değer; early stopping YOK
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
final_model.fit(
    X_full, y_full,
    categorical_feature=['sbc_category'],
)

LGBMRegressor(learning_rate=0.05, n_estimators=92, objective='tweedie',
              random_state=42, tweedie_variance_power=1.5, verbose=-1)

In [ ]:
mean52_final = (full_train.sort_values('week')
                           .groupby('product')['quantity']
                           .apply(lambda s: s.tail(52).mean()))

In [ ]:
import numpy as np

test_df_aligned = test_df.copy()
test_df_aligned['sbc_category'] = pd.Categorical(test_df_aligned['sbc_category'], categories=cat_categories_final)

X_test = test_df_aligned[FEATURES]
y_test = test_df_aligned['quantity']

test_preds = np.maximum(final_model.predict(X_test), 0)
test_baseline = test_df_aligned['product'].map(mean52_final).fillna(0).values

In [ ]:
test_pred_df = test_df_aligned[['product', 'week', 'sbc_category']].copy()
test_pred_df['actual'] = y_test.values
test_pred_df['pred'] = test_preds
test_pred_df['baseline'] = test_baseline
test_pred_df['block'] = ((test_pred_df['week'] - test_pred_df['week'].min()).dt.days // 28)

test_agg = test_pred_df.groupby(['product', 'block']).agg(
    pred=('pred', 'sum'),
    baseline=('baseline', 'sum'),
    actual=('actual', 'sum'),
    sbc_category=('sbc_category', 'first'),
)

In [ ]:
def wape(pred, actual):
    return np.abs(pred - actual).sum() / actual.sum()

def teknorot(pred, actual, under_penalty=2.0, over_penalty=1.0):
    pred = np.asarray(pred)
    actual = np.asarray(actual)
    under = np.maximum(actual - pred, 0)   # eksik tahmin tutarı
    over  = np.maximum(pred - actual, 0)   # fazla tahmin tutarı
    return (under_penalty * under.sum() + over_penalty * over.sum()) / actual.sum()

In [ ]:
test_agg['router'] = test_agg.apply(
    lambda row: row['pred'] if FINAL_DECISION.get(row['sbc_category'], 'baseline') == 'model' else row['baseline'],
    axis=1
)
test_agg['blend5050'] = 0.5 * test_agg['pred'] + 0.5 * test_agg['baseline']

method_labels = {'pred': 'Model', 'baseline': 'Baseline', 'router': 'Router', 'blend5050': '50/50 Blend'}

print("\n=== NİHAİ TEST SONUCU (tek seferlik) ===")
for m in ['pred', 'baseline', 'router', 'blend5050']:
    w = wape(test_agg[m], test_agg['actual'])
    t = teknorot(test_agg[m], test_agg['actual'])
    print(f"  {method_labels[m]:<12} WAPE={w:.3f}  Teknorot={t:.3f}")


=== NİHAİ TEST SONUCU (tek seferlik) ===
  Model        WAPE=0.890  Teknorot=1.362
  Baseline     WAPE=0.912  Teknorot=1.441
  Router       WAPE=0.879  Teknorot=1.370
  50/50 Blend  WAPE=0.894  Teknorot=1.392
